# 2 4 add correction

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


## All first corrections

In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=28).query('prior_corrections==0').reset_index(drop=True)
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from pystata import stata
stata.run('testparm treated_periods_pre*')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_cum', ylims=ylims, legend_ncol=1, legend_title='First correction')

In [ ]:
from utils import SEED
stata.run(
    f"""
    sdid tweet_count unitId weeks_from_display treated_after, ///
        vce(bootstrap) cluster(account_cluster) ///
        xline_opts(lcolor(none)) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xline(0, lcolor(black) lpattern(dash)) xtitle("Weeks from note display") ytitle("Post count") xlabel(-4(1)4) title("") graphregion(color(white)))
    """
)

In [ ]:
stata.run(f"graph export outputs/figures/sdid_posts_weekly_cum.pdf, replace")

### No treated accounts in control events

In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=28).query('prior_corrections==0').reset_index(drop=True)
treated_accounts = list(source_corrections.query('treated==1').user_id.unique())
source_corrections['treated_control'] = source_corrections.apply(lambda item: 1 if item['treated'] == 0 and item['user_id'] in treated_accounts else 0, axis=1)

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
len(ts_posts.query('treated_control==1').user_id.unique())

In [ ]:
stata_did = StataDiD(ts_posts.query('treated_control==0'), 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
from utils import get_coefs

stata_did.multi_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster')
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='qui ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='qui ppmlhdfe', dev='tweet_count', cluster='account_cluster')
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='qui ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)